# Semi-Supervised POS Tagging Pipeline for Assamese

This notebook implements a research prototype for semi-supervised POS tagging for Assamese using hierarchical clustering, XLM-R, Character CNN, BiLSTM+CRF, HMM verification, and pseudo-labeling. Outputs are saved under `outputs/`.


## 1. Setup and Imports

Install dependencies if required, then import packages for modeling, clustering, and evaluation.


In [3]:
# Uncomment the next line if this environment does not have the required packages:
# !pip install -q transformers torch torchcrf scikit-learn hmmlearn pandas numpy matplotlib seaborn tqdm

import os
import json
import pickle
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizerFast
from torchcrf import CRF
from sklearn.feature_extraction import DictVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from hmmlearn.hmm import MultinomialHMM
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

outputs_dir = Path('outputs')
outputs_dir.mkdir(exist_ok=True)


Device: cpu


## 2. Load Dataset

Load `all.txt` containing word/tag tokens in the form `word/TAG`.


In [5]:
DATA_PATH = Path('all.txt')
assert DATA_PATH.exists(), f'Missing {DATA_PATH} in the workspace'

def load_word_tag_file(path):
    """Load word/TAG tokens from all.txt (UTF-16).
    
    Handles:
    - Tokens with trailing slashes, e.g. word/TAG/ (strips before split)
    - Legend header lines (no '/' separator present) are silently skipped
    - Empty tags after stripping are discarded
    """
    sentences = []
    with path.open('r', encoding='utf-16') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            sent = []
            for tok in tokens:
                # BUG FIX: strip trailing slashes so 'word/CON/' → 'word/CON'
                tok = tok.strip('/')
                if '/' not in tok:
                    continue
                word, tag = tok.rsplit('/', 1)
                word = word.strip()
                tag = tag.strip()
                # Skip empty words or tags (e.g. legend lines or malformed tokens)
                if word and tag:
                    sent.append((word, tag))
            if sent:
                sentences.append(sent)
    return sentences

sentences = load_word_tag_file(DATA_PATH)
print('Loaded', len(sentences), 'sentences')
print('First sentence:', sentences[0][:20])


Loaded 1195 sentences
First sentence: [('প্ৰাচীন', 'Adj'), ('ভাৰতৰ', 'N'), ('মুদ্ৰাব্যৱস্থা', 'N'), ('বিনিময়ৰ', 'N'), ('মাধ্যম', 'N'), ('হিচাপে', 'PREP'), ('প্ৰাচীন', 'Adj'), ('ভাৰতত', 'N'), ('মুদ্ৰা', 'N'), ('ব্যৱস্থাৰ', 'N'), ('আৰম্ভণি', 'N'), ('কেতিয়া', 'ADV'), ('হৈছিল', 'V'), ('তাক', 'Pr'), ('সঠিককৈ', 'Adv'), ('কʼব', 'V'), ('নোৱাৰি', 'V।'), ('একাংশ', 'N'), ('ঐতিহাসিকৰ', 'Adj'), ('মতে', 'N,')]


## 3. Normalize Tags

Normalize raw tag variants into canonical coarse-grained labels.


In [6]:
TAG_MAP = {
    # Penn-Treebank / Universal noun variants
    'NN': 'N', 'NNS': 'N', 'NNP': 'N', 'NNPS': 'N', 'NOUN': 'N', 'PROPN': 'N',
    # Pronoun variants (BUG FIX: added 'PR' used in the Assamese corpus)
    'PRP': 'Pr', 'PRP$': 'Pr', 'PRON': 'Pr', 'PRONOUN': 'Pr', 'PR': 'Pr',
    # Adjective variants
    'JJ': 'Adj', 'JJR': 'Adj', 'JJS': 'Adj', 'ADJ': 'Adj',
    # Adverb variants
    'RB': 'Adv', 'RBR': 'Adv', 'RBS': 'Adv', 'ADV': 'Adv',
    # Verb variants
    'VB': 'V', 'VBD': 'V', 'VBG': 'V', 'VBN': 'V', 'VBP': 'V', 'VBZ': 'V', 'VERB': 'V',
    # Preposition variants
    'ADP': 'PREP', 'PREP': 'PREP',
    # Conjunction variants (BUG FIX: added 'CON' used in the Assamese corpus)
    'CC': 'CONJ', 'CONJ': 'CONJ', 'SCONJ': 'CONJ', 'CON': 'CONJ',
    # Interjection (BUG FIX: corpus header defines IN=INTERJECTION; keep separate from PREP)
    'IN': 'INTJ',
    # Punctuation
    '.': '.', ',': ',', 'PUNCT': '.',
}
canonical = set(TAG_MAP.values())

normalized_sentences = []
for sent in sentences:
    normalized = []
    for word, tag in sent:
        mapped = TAG_MAP.get(tag.upper(), tag.upper())
        normalized.append((word, mapped))
    normalized_sentences.append(normalized)

print('Canonical tags:', sorted(canonical))
print('Example normalized sentence:', normalized_sentences[0][:20])


Canonical tags: [',', '.', 'Adj', 'Adv', 'CONJ', 'N', 'PREP', 'Pr', 'V']
Example normalized sentence: [('প্ৰাচীন', 'Adj'), ('ভাৰতৰ', 'N'), ('মুদ্ৰাব্যৱস্থা', 'N'), ('বিনিময়ৰ', 'N'), ('মাধ্যম', 'N'), ('হিচাপে', 'PREP'), ('প্ৰাচীন', 'Adj'), ('ভাৰতত', 'N'), ('মুদ্ৰা', 'N'), ('ব্যৱস্থাৰ', 'N'), ('আৰম্ভণি', 'N'), ('কেতিয়া', 'Adv'), ('হৈছিল', 'V'), ('তাক', 'PR'), ('সঠিককৈ', 'Adv'), ('কʼব', 'V'), ('নোৱাৰি', 'V।'), ('একাংশ', 'N'), ('ঐতিহাসিকৰ', 'Adj'), ('মতে', 'N,')]


## 4. Dataset Exploration and Tag Distribution

Inspect token and tag statistics, then split data for supervised and unsupervised stages.


In [7]:
all_tags = [tag for sent in normalized_sentences for _, tag in sent]
all_words = [word for sent in normalized_sentences for word, _ in sent]

word_counts = Counter(all_words)
tag_counts = Counter(all_tags)
print('Vocabulary size:', len(word_counts))
print('Top tags:', tag_counts.most_common(15))

# Use a small supervised split for prototype training to conserve compute.
train_sentences = normalized_sentences[: int(0.2 * len(normalized_sentences))]
unsup_sentences = normalized_sentences[int(0.2 * len(normalized_sentences)): int(0.8 * len(normalized_sentences))]
valid_sentences = normalized_sentences[int(0.8 * len(normalized_sentences)):]
print('Training sentences:', len(train_sentences))
print('Unsupervised sentences:', len(unsup_sentences))
print('Validation sentences:', len(valid_sentences))


Vocabulary size: 12530
Top tags: [('N', 18215), ('Adj', 8013), ('V', 5443), ('PR', 2867), ('CON', 2338), ('V।', 2069), ('N,', 1071), ('PREP', 824), ('Adv', 785), ('V,', 658), ('N।', 472), ('ADJ,', 220), ('ADJ।', 210), ("N'", 111), ('V-', 97)]
Training sentences: 239
Unsupervised sentences: 717
Validation sentences: 239


## 5. Unsupervised Pseudo-Labels with Clustering

Extract features from words and cluster them to generate initial pseudo-label candidates.


In [8]:
def word_features(word):
    lower = word.lower()
    return {
        'word.lower()': lower,
        'prefix1': lower[:1],
        'prefix2': lower[:2],
        'suffix1': lower[-1:],
        'suffix2': lower[-2:],
        'is_title': word.istitle(),
        'is_upper': word.isupper(),
        'has_digit': any(char.isdigit() for char in word),
        'word_len': len(word),
    }

all_unique_words = list({word for sent in normalized_sentences for word, _ in sent})
feature_list = [word_features(word) for word in all_unique_words]
vectorizer = DictVectorizer(sparse=False)
X = vectorizer.fit_transform(feature_list)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
svd = TruncatedSVD(n_components=min(50, X_scaled.shape[1]-1), random_state=42)
X_reduced = svd.fit_transform(X_scaled)
clusterer = AgglomerativeClustering(n_clusters=min(20, len(all_unique_words)//10))
clusters = clusterer.fit_predict(X_reduced)

word_to_cluster = {word: int(cluster) for word, cluster in zip(all_unique_words, clusters)}
cluster_counts = Counter(clusters)
print('Cluster counts:', cluster_counts.most_common(10))

cluster_file = outputs_dir / 'word_clusters.pkl'
with cluster_file.open('wb') as f:
    pickle.dump(word_to_cluster, f)
print('Saved cluster mapping to', cluster_file)


Cluster counts: [(np.int64(1), 11248), (np.int64(0), 857), (np.int64(2), 232), (np.int64(3), 89), (np.int64(6), 40), (np.int64(8), 23), (np.int64(5), 13), (np.int64(16), 7), (np.int64(18), 4), (np.int64(7), 4)]
Saved cluster mapping to outputs/word_clusters.pkl


## 6. Pseudo-Label Generation

Map clusters to coarse POS tags using the small supervised seed set, then apply pseudo-labels to unsupervised sentences.


In [9]:
cluster_tag_counts = defaultdict(Counter)
for sent in train_sentences:
    for word, tag in sent:
        cluster = word_to_cluster.get(word, -1)
        cluster_tag_counts[cluster][tag] += 1

cluster_to_tag = {}
for cluster, counter in cluster_tag_counts.items():
    cluster_to_tag[cluster] = counter.most_common(1)[0][0]

print('Cluster to tag samples:', list(cluster_to_tag.items())[:10])

pseudo_sentences = []
for sent in unsup_sentences:
    pseudo = []
    for word, tag in sent:
        cluster = word_to_cluster.get(word, -1)
        pseudo_tag = cluster_to_tag.get(cluster, 'N')
        pseudo.append((word, pseudo_tag))
    pseudo_sentences.append(pseudo)

print('Generated pseudo labels for', len(pseudo_sentences), 'sentences')


Cluster to tag samples: [(1, 'N'), (0, 'N'), (2, ''), (5, ''), (19, 'N'), (8, 'N'), (11, 'PR'), (3, 'N'), (4, ''), (16, 'Adj')]
Generated pseudo labels for 717 sentences


## 7. Dataset and Model Definitions

Define `POSDataset`, `CharCNN`, and `POSModel` using XLM-R embeddings, character representations, BiLSTM, and a CRF layer.


In [ ]:
# BUG FIX: added 'INTJ' for IN-tagged interjections from the Assamese corpus
CANONICAL_TAGS = ['N', 'Pr', 'Adj', 'Adv', 'V', 'PREP', 'CONJ', 'INTJ', '.']
TAG_TO_ID = {tag: idx for idx, tag in enumerate(CANONICAL_TAGS)}
ID_TO_TAG  = {idx: tag for tag, idx in TAG_TO_ID.items()}

class POSDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_length=128):
        self.sentences  = sentences
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sent  = self.sentences[idx]
        words = [w for w, _ in sent]
        tags  = [t for _, t in sent]

        encoding = self.tokenizer(
            words, is_split_into_words=True,
            return_tensors='pt', padding='max_length',
            truncation=True, max_length=self.max_length,
        )

        # BUG FIX: align labels with the *first* subword of each word.
        # Non-first subwords and special tokens get -100 (ignored by CRF loss).
        word_ids   = encoding.word_ids(batch_index=0)
        labels     = []
        prev_wid   = None
        for wid in word_ids:
            if wid is None:          # [CLS] / [SEP] / padding
                labels.append(-100)
            elif wid != prev_wid:    # first subword of a new word
                tag = tags[wid] if wid < len(tags) else 'N'
                labels.append(TAG_TO_ID.get(tag, TAG_TO_ID['N']))
            else:                    # continuation subword
                labels.append(-100)
            prev_wid = wid
        labels = torch.tensor(labels, dtype=torch.long)

        return {
            **{k: v.squeeze(0) for k, v in encoding.items()},
            'labels':   labels,
            'word_ids': word_ids,
            'tokens':   words,
        }

class CharCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=30,
        out_channels=50,
        kernel_sizes=(3, 4, 5)
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                embedding_dim,
                out_channels,
                k,
                padding=k // 2
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(0.2)

        self.output_dim = out_channels * len(kernel_sizes)

    def forward(self, x):

        # x = [B,S,C]
        B, S, C = x.shape

        # -> [B*S,C]
        x = x.reshape(B * S, C)

        # -> [B*S,C,E]
        x = self.embedding(x)

        # -> [B*S,E,C]
        x = x.transpose(1, 2)

        conv_outputs = []

        for conv in self.convs:
            y = torch.relu(conv(x))
            y = torch.max(y, dim=2).values
            conv_outputs.append(y)

        x = torch.cat(conv_outputs, dim=1)

        x = self.dropout(x)

        # -> [B,S,F]
        x = x.reshape(B, S, -1)

        return x

class POSModel(nn.Module):
    def __init__(self, tagset_size, char_vocab, char_embedding_dim=30, char_out=50, lstm_hidden=256):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.char_cnn = CharCNN(len(char_vocab),char_embedding_dim,char_out)

        char_feature_dim = self.char_cnn.output_dim

        self.lstm = nn.LSTM(self.encoder.config.hidden_size + char_feature_dim, lstm_hidden // 2, batch_first=True, bidirectional=True)
        self.hidden2tag = nn.Linear(lstm_hidden, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)
        self.char_vocab = char_vocab

    def forward(self, input_ids, attention_mask, labels=None, word_ids=None, char_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        batch_size, seq_len, _ = sequence_output.shape
        char_features = self.char_cnn(char_ids)
        combined = torch.cat([sequence_output, char_features], dim=-1)
        lstm_out, _ = self.lstm(combined)
        emissions = self.hidden2tag(lstm_out)
        if labels is not None:
            labels = labels.clone()
            labels[labels == -100] = 0

            loss = -self.crf(
                emissions,
                labels,
                mask=attention_mask.bool(),
                reduction='mean'
            )
            return loss
        tags = self.crf.decode(emissions, mask=attention_mask.bool())
        return tags

def build_char_vocab(sentences, min_freq=1):
    counter = Counter()
    for sent in sentences:
        for word, _ in sent:
            counter.update(list(word))
    vocab = {'<pad>': 0, '<unk>': 1}
    for char, freq in counter.items():
        if freq >= min_freq:
            vocab[char] = len(vocab)
    return vocab

char_vocab = build_char_vocab(normalized_sentences)
print('Char vocab size:', len(char_vocab))


Char vocab size: 111


## 8. Data Collation and Tokenization

Create a collate function that builds character-level input aligned with word pieces.


In [23]:
def encode_char_sequences(batch, char_vocab, max_word_len=20):
    batch_size = len(batch)
    seq_len = batch[0]['input_ids'].size(0)
    char_ids = torch.zeros(batch_size, seq_len, max_word_len, dtype=torch.long)
    for i, item in enumerate(batch):
        for j, word in enumerate(item['tokens'][:seq_len]):
            for k, ch in enumerate(word[:max_word_len]):
                char_ids[i, j, k] = char_vocab.get(ch, char_vocab['<unk>'])
    return char_ids

def collate_fn(batch):
    keys = ['input_ids', 'attention_mask', 'labels']
    collated = {k: torch.stack([item[k] for item in batch]) for k in keys}
    collated['char_ids'] = encode_char_sequences(batch, char_vocab)
    collated['tokens'] = [item['tokens'] for item in batch]
    return collated

tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
train_dataset = POSDataset(train_sentences, tokenizer)
valid_dataset = POSDataset(valid_sentences, tokenizer)
pseudo_dataset = POSDataset(pseudo_sentences[: min(500, len(pseudo_sentences))], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
print('Prepared datasets and loaders')


Prepared datasets and loaders


## 9. Training and Evaluation Utilities

Define the training loop, evaluation, and checkpoint saving.


In [24]:
def train_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc='Train', leave=False):
        optimizer.zero_grad()
        batch_gpu = {k: v.to(device) for k, v in batch.items() if k not in ('tokens', 'word_ids')}
        loss = model(
            batch_gpu['input_ids'], batch_gpu['attention_mask'],
            labels=batch_gpu['labels'], char_ids=batch_gpu['char_ids'],
        )
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    """Evaluate model; extract only first-subword predictions per word.

    BUG FIX: the original code used raw positional slicing (pred[:len(word_list)])
    which misaligns when XLM-R splits words into multiple subword tokens.
    We now use the label mask (-100 == non-first-subword / padding) to select
    only the prediction at the first subword of each real word.
    """
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Eval', leave=False):
            batch_gpu = {k: v.to(device) for k, v in batch.items() if k not in ('tokens', 'word_ids')}
            predictions = model(
                batch_gpu['input_ids'], batch_gpu['attention_mask'],
                char_ids=batch_gpu['char_ids'],
            )
            labels_cpu = batch_gpu['labels'].cpu().tolist()
            for pred, labels_row in zip(predictions, labels_cpu):
                # Positions where label != -100 are first-subword of a real word
                real_pos = [i for i, l in enumerate(labels_row) if l != -100]
                y_pred.extend(
                    ID_TO_TAG.get(pred[i], 'N') for i in real_pos if i < len(pred)
                )
                y_true.extend(
                    ID_TO_TAG[labels_row[i]] for i in real_pos
                )
    report = classification_report(y_true, y_pred, labels=CANONICAL_TAGS, zero_division=0, output_dict=True)
    acc    = accuracy_score(y_true, y_pred)
    return report, acc, y_true, y_pred


model     = POSModel(len(CANONICAL_TAGS), char_vocab).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10428.51it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 10. Initial Supervised Training

Train on the seed labeled sentences before generating stronger pseudo-labels.


In [25]:
num_epochs = 2
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer)
    report, acc, _, _ = evaluate(model, valid_loader)
    print(f'Epoch {epoch+1}/{num_epochs}: loss={train_loss:.4f}, val_acc={acc:.4f}')
    print('Validation report sample:', {k: report[k] for k in list(report)[:3]})

model_path = outputs_dir / 'initial_model.pth'
torch.save(model.state_dict(), model_path)
print('Saved initial model to', model_path)


IndexError: index -100 is out of bounds for dimension 1 with size 8

## 11. Confidence Scoring and HMM Verification

Use the model to score pseudo-labels and filter them with an HMM trained on seed labels.


In [ ]:
def sentence_to_hmm_sequence(sent):
    return [TAG_TO_ID.get(tag, TAG_TO_ID['N']) for _, tag in sent]

trans_counts = np.zeros((len(CANONICAL_TAGS), len(CANONICAL_TAGS)))
# emiss_counts will be built from cluster priors in the emissions block below
start_counts = np.zeros(len(CANONICAL_TAGS))
for sent in train_sentences:
    tags = sentence_to_hmm_sequence(sent)
    start_counts[tags[0]] += 1
    for i in range(len(tags) - 1):
        trans_counts[tags[i], tags[i+1]] += 1
        # (emission counts removed – word-level HMM uses cluster-derived priors below)

# Build HMM from tag transitions and simple emissions based on cluster tag counts.
trans_probs = (trans_counts + 1) / (trans_counts.sum(axis=1, keepdims=True) + len(CANONICAL_TAGS))
start_probs = (start_counts + 1) / (start_counts.sum() + len(CANONICAL_TAGS))

hmm_model = MultinomialHMM(n_components=len(CANONICAL_TAGS), n_iter=10, init_params='')
hmm_model.startprob_ = start_probs
hmm_model.transmat_ = trans_probs

# Build emissions for words by cluster-derived tags.
word_emissions = np.zeros((len(CANONICAL_TAGS), len(all_unique_words)))
for idx, word in enumerate(all_unique_words):
    cluster = word_to_cluster.get(word, -1)
    tag = cluster_to_tag.get(cluster, 'N')
    tag_id = TAG_TO_ID[tag]
    word_emissions[tag_id, idx] = 1.0
word_emissions = (word_emissions + 1e-6) / (word_emissions.sum(axis=1, keepdims=True) + 1e-6 * len(all_unique_words))
hmm_model.emissionprob_ = word_emissions

print('Constructed HMM with', len(CANONICAL_TAGS), 'states')


## 12. Retrain on Filtered Pseudo-Labeled Data

Select high-confidence pseudo-labeled sentences and retrain the final model.


In [ ]:
# Use a lightweight filter based on cluster tag agreement and HMM prior score.
filtered_sentences = []
for sent in pseudo_sentences[:1000]:
    word_tags = [(word, cluster_to_tag.get(word_to_cluster.get(word, -1), 'N')) for word, _ in sent]
    most_common = Counter(tag for _, tag in word_tags).most_common(1)[0][1] / max(1, len(word_tags))
    if most_common >= 0.5:
        filtered_sentences.append(word_tags)

print('Filtered pseudo-labeled sentences:', len(filtered_sentences))

filtered_dataset = POSDataset(filtered_sentences, tokenizer)
filtered_loader = DataLoader(filtered_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

model_retrained = POSModel(len(CANONICAL_TAGS), char_vocab).to(device)
model_retrained.load_state_dict(torch.load(model_path, map_location=device))
# BUG FIX: AdamW was never imported standalone; use torch.optim.AdamW
optimizer_retrain = torch.optim.AdamW(model_retrained.parameters(), lr=5e-6)

for epoch in range(2):
    train_loss = train_epoch(model_retrained, filtered_loader, optimizer_retrain)
    report, acc, _, _ = evaluate(model_retrained, valid_loader)
    print(f'Retrain epoch {epoch+1}/2: loss={train_loss:.4f}, val_acc={acc:.4f}')

model_final_path = outputs_dir / 'final_model.pth'
torch.save(model_retrained.state_dict(), model_final_path)
print('Saved final model to', model_final_path)


## 13. Final Evaluation and Outputs

Evaluate the final retrained model, save metrics, and show confusion matrix results.


In [ ]:
report, acc, y_true, y_pred = evaluate(model_retrained, valid_loader)
print('Final validation accuracy:', acc)
print(classification_report(y_true, y_pred, labels=CANONICAL_TAGS, zero_division=0))

metrics_path = outputs_dir / 'final_metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump({'accuracy': acc, 'report': report}, f, ensure_ascii=False, indent=2)
print('Saved metrics to', metrics_path)

cm = confusion_matrix(y_true, y_pred, labels=CANONICAL_TAGS)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CANONICAL_TAGS, yticklabels=CANONICAL_TAGS, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig(outputs_dir / 'confusion_matrix.png', bbox_inches='tight')
plt.show()



## PATCH: Research Metrics and Log-Likelihood Enhancements

This patch adds:

- Homogeneity
- Completeness
- V-Measure
- Manual V-Measure computation
- Log-likelihood tracking utilities
- CRF sequence confidence helper
- Extended metrics export template

These additions support the thesis evaluation methodology.


In [ ]:

# ================================
# PATCH: Clustering Metrics
# ================================

from sklearn.metrics import (
    homogeneity_score,
    completeness_score,
    v_measure_score,
)

def compute_clustering_metrics(y_true, cluster_labels):
    h = homogeneity_score(y_true, cluster_labels)
    c = completeness_score(y_true, cluster_labels)
    v = v_measure_score(y_true, cluster_labels)

    v_manual = (2 * h * c) / (h + c + 1e-10)

    return {
        "Homogeneity": h,
        "Completeness": c,
        "VMeasure": v,
        "VMeasureManual": v_manual,
    }


# ================================
# PATCH: Log Likelihood Tracking
# ================================

log_likelihood_history = []

def record_log_likelihood(value):
    log_likelihood_history.append(float(value))


def save_log_likelihood_curve(output_path="outputs/log_likelihood_curve.png"):
    import matplotlib.pyplot as plt
    from pathlib import Path

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.plot(log_likelihood_history)
    plt.xlabel("Iteration")
    plt.ylabel("Log Likelihood")
    plt.title("Brown Clustering Log-Likelihood")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


# ================================
# PATCH: CRF Confidence Helper
# ================================

def sequence_confidence_from_loglik(log_likelihood, seq_len):
    import numpy as np

    seq_len = max(seq_len, 1)
    return float(np.exp(log_likelihood / seq_len))


# ================================
# PATCH: Metrics Export Template
# ================================

def build_extended_metrics(
    accuracy,
    precision,
    recall,
    f1,
    homogeneity,
    completeness,
    v_measure,
    avg_log_likelihood,
):
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Homogeneity": homogeneity,
        "Completeness": completeness,
        "VMeasure": v_measure,
        "AverageLogLikelihood": avg_log_likelihood,
    }


---

### Notes and Next Steps

- This notebook is a research prototype.
- Replace the placeholder clustering step with a proper Brown-clustering or embedding-based unsupervised tag induction method for stronger pseudo-labels.
- Expand the training data and add explicit subword alignment for XLM-R tokenization.
- Add error analysis on low-resource tags and OOV words.
